# MammoRisk+ Module 3: train the benign vs malignant classifier on Kaggle GPU (O2)

Trains **one** backbone per run on the cached CBIS-DDSM full mammograms and then evaluates it once on
val/test. Run it twice: `MODEL = "efficientnet_b3"` and again with `MODEL = "resnet50"` (the baseline),
then compare the two with the DeLong test (last cells).

**Settings before running:**
1. Notebook options → **Accelerator: GPU** (T4 x2 or P100) and **Internet: On** (clone repo, download ImageNet weights).
2. **Add Input** → your private dataset `mammorisk-cbis-png-1024x512` (built by `m3_cbis_cache_kaggle.ipynb`).
3. **Save Version → Save & Run All** so it keeps running in the background. A GPU session is limited to ~9-12 h.

Reproducibility (CLAUDE.md): seed 42 (set inside `train`), one YAML config per run (the exact overridden
YAML is saved next to the checkpoint), split fixed in the cache manifest (`splits/m3_cbis_split.csv`),
patient-level, augmentation on train only. The test set is used **once**, in the evaluate cell, with the
threshold chosen on val (Youden).

In [ ]:
MODEL = "efficientnet_b3"      # "efficientnet_b3" or "resnet50" (run the notebook once per model)
WANDB = False                   # True: add a Kaggle secret named WANDB_API_KEY (Add-ons -> Secrets)

In [ ]:
!git clone --depth 1 https://github.com/FaiqaZarar/MammoRisk-Plus.git /kaggle/working/code
%cd /kaggle/working/code
!git log --oneline -1
!pip -q install timm wandb
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("NO GPU: turn on the accelerator (Notebook options); training on CPU would take days")

## Find the cache and write the run config
The base config (`configs/m3_<model>.yaml`) is loaded and only the **paths** (and the worker count) are changed
for Kaggle. Everything else (lr, batch size, epochs, augmentation) stays as in the repo, so this is the same
experiment as on any other machine. The merged YAML is saved in the output.

In [ ]:
import glob, os, yaml, pandas as pd
hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
assert len(hits) == 1, f"Expected exactly one manifest.csv under /kaggle/input, found: {hits}"
CACHE = os.path.dirname(hits[0])
print("cache:", CACHE)

base = f"configs/m3_{'effnetb3' if MODEL == 'efficientnet_b3' else 'resnet50'}.yaml"
cfg = yaml.safe_load(open(base))
assert cfg["model"]["name"] == MODEL
cfg["paths"]["cache_dir"] = CACHE
cfg["paths"]["results_dir"] = "/kaggle/working/results/m3"
cfg["train"]["num_workers"] = 4              # Kaggle has 4 CPU cores
cfg["wandb"]["enabled"] = WANDB
os.makedirs("/kaggle/working/results/m3", exist_ok=True)
RUN_CFG = f"/kaggle/working/results/m3/{cfg['run_name']}_kaggle.yaml"
yaml.safe_dump(cfg, open(RUN_CFG, "w"), sort_keys=False)
print("run config:", RUN_CFG, "| run:", cfg["run_name"])

if WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Sanity check before the long run
Confirms the cache matches the committed split (counts, patient-level separation, class balance).
**Expected (D-20):** train 2,138 / val 291 / test 603 images; 1,095 / 157 / 314 patients (fewer if some
images failed in the cache build; check the status counts).

In [ ]:
man = pd.read_csv(f"{CACHE}/manifest.csv")
print(man.status.value_counts().to_dict())
ok = man[man.status == "ok"]
print(ok.groupby("split").agg(images=("image_id", "size"), patients=("patient_id", "nunique"),
                              malignant_frac=("label", "mean")).round(3))
sets = {s: set(ok[ok.split == s].patient_id) for s in ("train", "val", "test")}
assert not (sets["train"] & sets["val"] or sets["train"] & sets["test"] or sets["val"] & sets["test"]), "PATIENT LEAKAGE"
print("no patient appears in more than one split")
repo_split = pd.read_csv("splits/m3_cbis_split.csv").set_index("image_id")["split"]
assert (ok.set_index("image_id")["split"] == repo_split.reindex(ok.image_id).values).all(), "cache split != committed split"
print("cache split matches splits/m3_cbis_split.csv")

## Train
Weighted cross-entropy, AdamW + warm-up + cosine, AMP, effective batch 32 (8 x 4 accumulation), early stopping on
**val AUC** (patience 10, max 50 epochs). The best-val-AUC checkpoint is saved as `best.pt`.
There is no resume: if the session dies, the run restarts from scratch. Check the epoch time printed below
and make sure 50 epochs fit in the session limit.

In [ ]:
!python -W ignore -m module3_classifier.train --config {RUN_CFG}

In [ ]:
import matplotlib.pyplot as plt
RUN = cfg["run_name"]; OUT = f"/kaggle/working/results/m3/{RUN}"
h = pd.read_csv(f"{OUT}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch, h.val_auc); ax[1].axvline(h.val_auc.idxmax(), ls="--", c="r")
ax[1].set_title(f"val AUC (best {h.val_auc.max():.3f} @ epoch {h.val_auc.idxmax()})"); ax[1].set_xlabel("epoch")
plt.show()
print(open(f"{OUT}/train_summary.json").read())

## Evaluate (uses the test set ONCE)
Threshold = Youden J on **val**, applied unchanged to test. AUC with DeLong and patient-clustered bootstrap
95% CI, plus sensitivity, specificity, F1, PPV. The CMMD/DMID external sets are skipped here (their caches are
not built yet), which `evaluation.json` records. Do not tune anything on the basis of the test result.

In [ ]:
!python -W ignore -m module3_classifier.evaluate run --config {RUN_CFG} --checkpoint {OUT}/best.pt

In [ ]:
import json
rep = json.load(open(f"{OUT}/evaluation.json"))
print("model:", rep["model"], "| best epoch:", rep["best_epoch"], "| config hash:", rep["config_hash"])
print("threshold:", rep["threshold"])
for k in ("CBIS-DDSM/val", "CBIS-DDSM/test"):
    print("\n" + k); print(json.dumps(rep[k], indent=2))
print("\ntest AUC (target >= 0.80):", rep["CBIS-DDSM/test"]["auc_delong_estimate"])

## Compare the two backbones (after BOTH models have been trained)
Paired DeLong test on the same test images. Both `predictions_test.csv` files must exist. Either train both in
one session, or add the first run's saved output as an Input and edit the two paths.

In [ ]:
A = "/kaggle/working/results/m3/m3_effnetb3_seed42/predictions_test.csv"
B = "/kaggle/working/results/m3/m3_resnet50_seed42/predictions_test.csv"
if os.path.exists(A) and os.path.exists(B):
    !python -W ignore -m module3_classifier.evaluate compare --pred-a {A} --pred-b {B}
else:
    print("Need both predictions_test.csv files; train the other model first (or add its output as an Input).")

## Save the results
**Save Version → Save & Run All** keeps `/kaggle/working/results/m3/` as the notebook output: `best.pt`,
`history.csv`, `train_summary.json`, `evaluation.json`, predictions and the run YAML. Record the numbers in
`docs/STATUS.md` (dataset, split, checkpoint, config, run) as CLAUDE.md requires. Model weights are never committed.

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/code
!du -sh /kaggle/working/results/m3/*